# SVM - Real Estate Price Prediction (Florida)

Entrena un modelo SVM (Support Vector Regression) sobre el dataset con feature engineering aplicado.  
Usa **Optuna** para busqueda bayesiana de hiperparametros y **KFold estratificado** para evaluacion OOF.  
Incluye preprocesamiento con **StandardScaler** e **imputacion de nulos** (requeridos por SVM).

**Flujo:**
1. Imports & Config  
2. Carga de datos (`train_fe.parquet`, `test_fe.parquet`)  
3. Definicion de features  
4. Preprocesamiento (imputacion + encoding + escalado)  
5. Busqueda de hiperparametros con Optuna  
6. Entrenamiento KFold con mejores parametros  
7. Evaluacion OOF  
8. Predicciones de test  
9. Exportacion de submission — `submissions/Pippin_03.00.csv`  
10. Exportacion Train — `submissions/test/Pippin_03.00.csv`  
11. Exportacion para metamodelo — test holdout (`data/meta/test_svm.csv`)  
12. Predicciones sobre competition_test  
13. Exportacion para metamodelo — competition test (`data/meta/comp_svm.csv`)

---
## 1. Imports & Config

In [ ]:
from __future__ import annotations

import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import optuna

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

from sklearn.svm import SVR
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import mean_absolute_percentage_error

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

# Paths
ROOT        = Path('../..')
DATA_DIR    = ROOT / 'data' / 'tabular'
SUBMISSIONS = ROOT / 'submissions'
TRAIN_PATH  = DATA_DIR / 'train_fe.parquet'
TEST_PATH   = DATA_DIR / 'test_fe.parquet'

# Experimento
SUBMISSION_NAME = 'Pippin_03.00'
STUDY_NAME      = f'svm_{SUBMISSION_NAME}'  # nombre unico del estudio Optuna

# Columnas de control
ID_COL     = 'zpid'
TARGET     = 'log_price'                   # se entrena en escala log
TARGET_RAW = 'lastSoldPrice_hpi_adjusted'  # precio real (solo para metricas)
STRAT_COL  = 'price_bin'                   # bin para StratifiedKFold

# KFold
SEED    = 42
N_FOLDS = 5

# Optuna
OPTUNA_TRIALS = 30   # numero de trials de busqueda (SVM es mas lento que LGBM)
OPTUNA_FOLDS  = 3    # folds internos en optuna

np.random.seed(SEED)
print('Config OK')

---
## 2. Carga de Datos

In [ ]:
df_train = pd.read_parquet(TRAIN_PATH)
df_test  = pd.read_parquet(TEST_PATH)

print(f'Train : {df_train.shape}')
print(f'Test  : {df_test.shape}')
print()
print('Target (log_price) stats:')
print(df_train[TARGET].describe().round(4))

---
## 3. Definicion de Features

Se excluyen columnas de identidad, target, estratificacion y texto libre.  
A diferencia de LightGBM, **SVM requiere**:
- Imputacion explicita de nulos
- Encoding de variables categoricas
- Escalado de features (StandardScaler)

| Grupo | Tipo | Tratamiento |
|---|---|---|
| `homeType` | `object` | OrdinalEncoder + imputacion por moda |
| `decade`, `era`, `school_rating_tier` | `Int64` nullable | Convertir a `float64` + imputacion por mediana |
| Resto numerico | `float64` / `int64` | Imputacion por mediana + StandardScaler |

In [ ]:
EXCLUDE = {ID_COL, TARGET, TARGET_RAW, STRAT_COL, 'description'}

FEATURE_COLS = [c for c in df_train.columns if c not in EXCLUDE]

CAT_COLS = ['homeType']

# Int64 nullable -> float64
INT64_NULLABLE = [c for c in FEATURE_COLS
                  if df_train[c].dtype == pd.Int64Dtype()]

for df in [df_train, df_test]:
    for col in INT64_NULLABLE:
        df[col] = df[col].astype(float)

NUM_COLS = [c for c in FEATURE_COLS if c not in CAT_COLS]

print(f'Features totales : {len(FEATURE_COLS)}')
print(f'Cat cols         : {CAT_COLS}')
print(f'Num cols         : {len(NUM_COLS)}')
print(f'Int64 convertidos: {INT64_NULLABLE}')

remaining = [c for c in FEATURE_COLS if df_train[c].dtype == pd.Int64Dtype()]
assert not remaining, f'Quedan Int64 nullable: {remaining}'
print('Dtypes OK')

In [ ]:
X_train_raw = df_train[FEATURE_COLS].copy()
y_train     = df_train[TARGET].copy()
strat       = df_train[STRAT_COL].astype(str)  # etiquetas para StratifiedKFold

X_test_raw  = df_test[FEATURE_COLS].copy()

# Si test tiene labels (holdout con targets conocidos)
has_test_labels = TARGET_RAW in df_test.columns
y_test_raw  = df_test[TARGET_RAW].values if has_test_labels else None
y_train_raw = df_train[TARGET_RAW].values

print(f'X_train_raw : {X_train_raw.shape}')
print(f'X_test_raw  : {X_test_raw.shape}')
print(f'Test tiene labels: {has_test_labels}')

---
## 4. Preprocesamiento

Construccion del pipeline de preprocesamiento con `ColumnTransformer`:
- **Numericas**: `SimpleImputer(median)` → `StandardScaler`
- **Categoricas**: `SimpleImputer(most_frequent)` → `OrdinalEncoder`

El preprocesador se ajusta **solo sobre el train** de cada fold para evitar data leakage.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler()),
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)),
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, NUM_COLS),
        ('cat', categorical_transformer, CAT_COLS),
    ],
    remainder='drop',
)

print('Preprocessor OK')
print(f'  Numericas  : {len(NUM_COLS)} cols → impute(median) + StandardScaler')
print(f'  Categoricas: {len(CAT_COLS)} cols → impute(mode)   + OrdinalEncoder')

---
## 5. Busqueda de Hiperparametros con Optuna

Busqueda bayesiana con `TPESampler` sobre el espacio de parametros de SVR.  
La metrica de optimizacion es **MAPE** sobre los precios reales (`expm1` del target log).

| Parametro | Rango | Razon |
|---|---|---|
| `C` | 0.01 - 1000 log | Penalizacion: controla margen vs error de entrenamiento |
| `epsilon` | 1e-4 - 1.0 log | Zona de tolerancia: cuanto error se acepta sin penalizar |
| `gamma` | 1e-5 - 1.0 log | Influencia de cada punto (kernel RBF) |
| `kernel` | rbf, linear | rbf captura no linealidades; linear es mas rapido |

In [ ]:
def optuna_objective(trial):
    kernel  = trial.suggest_categorical('kernel', ['rbf', 'linear'])
    C       = trial.suggest_float('C', 1e-2, 1e3, log=True)
    epsilon = trial.suggest_float('epsilon', 1e-4, 1.0, log=True)

    params = dict(kernel=kernel, C=C, epsilon=epsilon)

    # gamma solo aplica para kernel RBF
    if kernel == 'rbf':
        gamma = trial.suggest_float('gamma', 1e-5, 1.0, log=True)
        params['gamma'] = gamma
    else:
        params['gamma'] = 'auto'

    skf    = StratifiedKFold(n_splits=OPTUNA_FOLDS, shuffle=True, random_state=SEED)
    scores = []

    for tr_idx, val_idx in skf.split(X_train_raw, strat):
        X_tr_raw  = X_train_raw.iloc[tr_idx]
        X_val_raw = X_train_raw.iloc[val_idx]
        y_tr      = y_train.iloc[tr_idx]
        y_val     = y_train.iloc[val_idx]

        # Ajustar preprocesador SOLO sobre train del fold (sin leakage)
        prep_fold = ColumnTransformer(
            transformers=[
                ('num', Pipeline([('imputer', SimpleImputer(strategy='median')),
                                  ('scaler',  StandardScaler())]), NUM_COLS),
                ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')),
                                  ('encoder', OrdinalEncoder(
                                      handle_unknown='use_encoded_value',
                                      unknown_value=-1))]), CAT_COLS),
            ],
            remainder='drop',
        )

        X_tr  = prep_fold.fit_transform(X_tr_raw)
        X_val = prep_fold.transform(X_val_raw)

        model = SVR(**params)
        model.fit(X_tr, y_tr.values)

        preds = model.predict(X_val)
        mape  = mean_absolute_percentage_error(
            np.expm1(y_val), np.expm1(preds)
        )
        scores.append(mape)

    return float(np.mean(scores))


print(f'Lanzando Optuna: {OPTUNA_TRIALS} trials x {OPTUNA_FOLDS} folds internos...')

In [ ]:
study = optuna.create_study(
    study_name=STUDY_NAME,
    direction='minimize',
    sampler=optuna.samplers.TPESampler(seed=SEED),
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=5),
    load_if_exists=True,
)

study.optimize(optuna_objective, n_trials=OPTUNA_TRIALS, show_progress_bar=True)

best_params_optuna = study.best_params
best_mape_optuna   = study.best_value

print(f'Estudio: {study.study_name}  |  trials completados: {len(study.trials)}')
print(f'\nMejor MAPE Optuna ({OPTUNA_FOLDS}-fold interno): '
      f'{best_mape_optuna:.4f}  ({best_mape_optuna*100:.2f}%)')
print('\nMejores hiperparametros:')
for k, v in best_params_optuna.items():
    print(f'  {k:<22} = {v}')

In [ ]:
# Visualizacion del estudio Optuna
try:
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    # Curva de optimizacion
    trial_values = [t.value for t in study.trials if t.value is not None]
    best_so_far  = np.minimum.accumulate(trial_values)
    axes[0].plot(trial_values, 'o', alpha=0.4, color='steelblue', ms=4, label='Trial')
    axes[0].plot(best_so_far, '-', color='crimson', lw=2, label='Mejor acumulado')
    axes[0].set_xlabel('Trial')
    axes[0].set_ylabel('MAPE')
    axes[0].set_title('Optuna — Curva de optimizacion')
    axes[0].legend()

    # Importancia de hiperparametros
    try:
        importance = optuna.importance.get_param_importances(study)
        imp_df = pd.DataFrame(list(importance.items()),
                              columns=['param', 'importance']).sort_values('importance')
        axes[1].barh(imp_df['param'], imp_df['importance'], color='steelblue')
        axes[1].set_xlabel('Importancia relativa')
        axes[1].set_title('Importancia de Hiperparametros (FAnova)')
    except Exception as e:
        axes[1].text(0.5, 0.5, f'No disponible: {e}',
                     ha='center', va='center', transform=axes[1].transAxes)

    plt.tight_layout()
    plt.show()
except Exception as e:
    print(f'Visualizacion no disponible: {e}')

---
## 6. Entrenamiento KFold con Mejores Parametros

`StratifiedKFold` sobre `price_bin` garantiza que cada fold tenga la misma distribucion de precios.  
El preprocesador se reajusta en cada fold para evitar data leakage.  
Las predicciones de test se promedian sobre los `N_FOLDS` modelos (ensemble de folds).

In [ ]:
# Parametros finales: mejores de Optuna
FINAL_PARAMS = dict(**best_params_optuna)

print('Parametros finales del modelo (SVR):')
for k, v in FINAL_PARAMS.items():
    print(f'  {k:<22} = {v}')

In [ ]:
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

oof_log_preds  = np.zeros(len(X_train_raw))  # predicciones OOF en escala log
test_log_preds = np.zeros(len(X_test_raw))   # predicciones test (promedio folds)
fold_mapes     = []
fold_preps     = []   # guardamos los preprocesadores ajustados
fold_models    = []   # guardamos los modelos

print(f'KFold {N_FOLDS} folds - StratifiedKFold sobre price_bin\n')

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train_raw, strat)):
    X_tr_raw  = X_train_raw.iloc[tr_idx]
    X_val_raw = X_train_raw.iloc[val_idx]
    y_tr      = y_train.iloc[tr_idx]
    y_val     = y_train.iloc[val_idx]

    # Preprocesador ajustado solo sobre el train del fold
    prep_fold = ColumnTransformer(
        transformers=[
            ('num', Pipeline([('imputer', SimpleImputer(strategy='median')),
                              ('scaler',  StandardScaler())]), NUM_COLS),
            ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')),
                              ('encoder', OrdinalEncoder(
                                  handle_unknown='use_encoded_value',
                                  unknown_value=-1))]), CAT_COLS),
        ],
        remainder='drop',
    )

    X_tr  = prep_fold.fit_transform(X_tr_raw)
    X_val = prep_fold.transform(X_val_raw)
    X_te  = prep_fold.transform(X_test_raw)

    model = SVR(**FINAL_PARAMS)
    model.fit(X_tr, y_tr.values)

    oof_log_preds[val_idx]  = model.predict(X_val)
    test_log_preds         += model.predict(X_te) / N_FOLDS

    fold_preps.append(prep_fold)
    fold_models.append(model)

    fold_mape = mean_absolute_percentage_error(
        np.expm1(y_val), np.expm1(oof_log_preds[val_idx])
    )
    fold_mapes.append(fold_mape)
    print(f'  Fold {fold+1}/{N_FOLDS} '
          f'| MAPE={fold_mape:.4f} ({fold_mape*100:.2f}%)')

print(f'\nMAPE OOF promedio : {np.mean(fold_mapes):.4f} ({np.mean(fold_mapes)*100:.2f}%)')
print(f'MAPE OOF std      : {np.std(fold_mapes):.4f}')

---
## 7. Evaluacion OOF

In [ ]:
oof_price_preds = np.expm1(oof_log_preds)

oof_mape = mean_absolute_percentage_error(y_train_raw, oof_price_preds)
oof_rmse = np.sqrt(np.mean((oof_log_preds - y_train.values) ** 2))
oof_mae  = np.mean(np.abs(oof_price_preds - y_train_raw))

print('=== Metricas OOF (train completo) ===')
print(f'  MAPE      : {oof_mape:.4f}  ({oof_mape*100:.2f}%)')
print(f'  RMSE(log) : {oof_rmse:.4f}')
print(f'  MAE       : ${oof_mae:,.0f} USD')
print()

# MAPE por bin de precio
eval_df = pd.DataFrame({
    'real' : y_train_raw,
    'pred' : oof_price_preds,
    'bin'  : df_train[STRAT_COL].values,
})
eval_df['pct_err'] = np.abs(eval_df['pred'] - eval_df['real']) / eval_df['real']

print('MAPE por bin de precio:')
print(eval_df.groupby('bin')['pct_err'].mean().apply(lambda x: f'{x*100:.2f}%').to_string())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Real vs Predicho
rng    = np.random.default_rng(SEED)
sample = rng.choice(len(y_train_raw), size=min(3000, len(y_train_raw)), replace=False)
mn     = min(y_train_raw.min(), oof_price_preds.min()) / 1e6
mx     = max(y_train_raw.max(), oof_price_preds.max()) / 1e6

axes[0].scatter(y_train_raw[sample] / 1e6, oof_price_preds[sample] / 1e6,
                alpha=0.3, s=6, color='steelblue')
axes[0].plot([mn, mx], [mn, mx], 'r--', lw=1.5, label='Perfecto')
axes[0].set_xlabel('Precio real (M USD)')
axes[0].set_ylabel('Precio predicho (M USD)')
axes[0].set_title(f'OOF: Real vs Predicho  (MAPE={oof_mape*100:.2f}%)')
axes[0].legend()

# Distribucion del error porcentual
pct_err = (oof_price_preds - y_train_raw) / y_train_raw * 100
axes[1].hist(pct_err, bins=80, color='steelblue', edgecolor='none', alpha=0.8)
axes[1].axvline(0, color='red', lw=1.5, ls='--')
axes[1].set_xlabel('Error porcentual (%)')
axes[1].set_ylabel('Frecuencia')
axes[1].set_title('Distribucion del Error Porcentual (OOF)')

# MAPE por fold
axes[2].bar(range(1, N_FOLDS + 1), [m * 100 for m in fold_mapes],
            color='steelblue', alpha=0.8, edgecolor='white')
axes[2].axhline(np.mean(fold_mapes) * 100, color='red', lw=1.5, ls='--', label='Promedio')
axes[2].set_xlabel('Fold')
axes[2].set_ylabel('MAPE (%)')
axes[2].set_title('MAPE por Fold')
axes[2].legend()

plt.tight_layout()
plt.show()

---
## 8. Predicciones de Test (Holdout 20%)

In [ ]:
test_price_preds = np.expm1(test_log_preds)

print(f'Predicciones de test: {len(test_price_preds):,} filas')
print(f'  Min    : ${test_price_preds.min():>12,.0f}')
print(f'  Media  : ${test_price_preds.mean():>12,.0f}')
print(f'  Mediana: ${np.median(test_price_preds):>11,.0f}')
print(f'  Max    : ${test_price_preds.max():>12,.0f}')

if has_test_labels:
    test_mape = mean_absolute_percentage_error(y_test_raw, test_price_preds)
    print(f'\nMAPE en test holdout: {test_mape:.4f} ({test_mape*100:.2f}%)')

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(test_price_preds / 1e6, bins=60,
        color='steelblue', edgecolor='none', alpha=0.8, label='Pred. test')
ax.hist(y_train_raw / 1e6, bins=60,
        color='orange', edgecolor='none', alpha=0.5, label='Real train')
ax.set_xlabel('Precio (M USD)')
ax.set_ylabel('Frecuencia')
ax.set_title('Distribucion: Predicciones Test vs Train Real')
ax.legend()
plt.tight_layout()
plt.show()

---
## 9. Exportacion de Submission

Formato requerido: `zpid, predicted_price`  
Los precios se exportan en escala real (no logaritmica), redondeados a 2 decimales.

In [ ]:
SUBMISSIONS.mkdir(parents=True, exist_ok=True)

submission = pd.DataFrame({
    'zpid'            : df_test[ID_COL].values,
    'predicted_price' : test_price_preds.round(2),
})

out_path = SUBMISSIONS / f'{SUBMISSION_NAME}.csv'
submission.to_csv(out_path, index=False)

print(f'Submission guardada en: {out_path}')
print(f'Filas: {len(submission):,}')
print()
print(submission.head(10).to_string(index=False))

In [ ]:
# Verificacion del formato de submission
check = pd.read_csv(out_path)
assert list(check.columns) == ['zpid', 'predicted_price'], 'Columnas incorrectas'
assert check['predicted_price'].notna().all(), 'Hay NaN en predicciones'
assert (check['predicted_price'] > 0).all(), 'Hay precios negativos o cero'

print('=== Resumen Final ===')
print(f'  Archivo           : {out_path.name}')
print(f'  Filas             : {len(check):,}')
print(f'  MAPE OOF (train)  : {oof_mape*100:.2f}%')
if has_test_labels:
    print(f'  MAPE holdout test : {test_mape*100:.2f}%')
print(f'  Precio mediano    : ${check["predicted_price"].median():,.0f}')
print(f'  Precio minimo     : ${check["predicted_price"].min():,.0f}')
print(f'  Precio maximo     : ${check["predicted_price"].max():,.0f}')
print()
print('Submission OK')

---
## 10. Exportacion Train — submissions/test/

Combina las predicciones OOF del train (9 472 filas) con las predicciones del holdout (2 368 filas)
y las guarda en `submissions/test/<SUBMISSION_NAME>.csv` con el mismo formato que la submission oficial.

In [ ]:
TEST_SUBMISSIONS = SUBMISSIONS / 'test'
TEST_SUBMISSIONS.mkdir(parents=True, exist_ok=True)

train_export = pd.DataFrame({
    'zpid'            : df_train[ID_COL].values,
    'predicted_price' : oof_price_preds.round(2),
})

holdout_export = pd.DataFrame({
    'zpid'            : df_test[ID_COL].values,
    'predicted_price' : test_price_preds.round(2),
})

train_full_export = pd.concat([train_export, holdout_export], ignore_index=True)

train_out_path = TEST_SUBMISSIONS / f'{SUBMISSION_NAME}.csv'
train_full_export.to_csv(train_out_path, index=False)

check_train = pd.read_csv(train_out_path)
assert list(check_train.columns) == ['zpid', 'predicted_price'], 'Columnas incorrectas'
assert check_train['predicted_price'].notna().all(), 'Hay NaN en predicciones'
assert (check_train['predicted_price'] > 0).all(), 'Hay precios negativos o cero'

print(f'=== Train Export ===')
print(f'  Archivo        : {train_out_path}')
print(f'  Filas train    : {len(train_export):,}  (OOF)')
print(f'  Filas holdout  : {len(holdout_export):,}')
print(f'  Total          : {len(check_train):,}')
print(f'  Precio mediano : ${check_train["predicted_price"].median():,.0f}')
print(f'  Precio minimo  : ${check_train["predicted_price"].min():,.0f}')
print(f'  Precio maximo  : ${check_train["predicted_price"].max():,.0f}')
print()
print(check_train.head(5).to_string(index=False))

---
## 11. Exportacion para Metamodelo — Test Holdout

Guarda las predicciones sobre el test holdout en `data/meta/test_svm.csv` para usarlas
como feature de entrada en el metamodelo (stacking).

Las predicciones se guardan en **escala logaritmica** (`log_price`) para que el metamodelo
opere en la misma escala que el target de entrenamiento.

In [ ]:
META_DIR = ROOT / 'data' / 'meta'
META_DIR.mkdir(parents=True, exist_ok=True)

meta_test = pd.DataFrame({
    'zpid'     : df_test[ID_COL].values,
    'svm_pred' : test_log_preds.round(6),
})

meta_test_path = META_DIR / 'test_svm.csv'
meta_test.to_csv(meta_test_path, index=False)

print(f'Exportado: {meta_test_path}')
print(f'Filas: {len(meta_test):,}')
print(meta_test.head())

---
## 12. Predicciones sobre competition_test_fe.parquet

Carga el parquet del conjunto de competencia real, aplica las mismas
transformaciones de tipo que al train/test, y genera predicciones promediando los `N_FOLDS` modelos.

> Los modelos del KFold ya estan entrenados en memoria. Si se reinicia el kernel,
> hay que volver a ejecutar las celdas anteriores antes de correr esta seccion.

In [ ]:
SUBMISSION_FE_PATH = DATA_DIR / 'competition_test_fe_sent.parquet'

df_submission = pd.read_parquet(SUBMISSION_FE_PATH)
print(f'df_competition_test_fe shape: {df_submission.shape}')
print(f'Columnas presentes  : {len(df_submission.columns)}')
print(f'zpid sample         : {df_submission[ID_COL].head(3).tolist()}')

# Verificar que tiene todas las features esperadas
missing_cols = [c for c in FEATURE_COLS if c not in df_submission.columns]
if missing_cols:
    print(f'WARN - Features faltantes ({len(missing_cols)}): {missing_cols}')
else:
    print(f'Features OK - todas las {len(FEATURE_COLS)} columnas presentes')

In [ ]:
# Aplicar las mismas transformaciones de tipo que al train/test
for col in INT64_NULLABLE:
    if col in df_submission.columns:
        df_submission[col] = df_submission[col].astype(float)

X_submission_raw = df_submission[FEATURE_COLS].copy()
print(f'X_submission_raw : {X_submission_raw.shape}')

In [ ]:
# Promediar predicciones usando los N_FOLDS modelos ya entrenados
# Cada modelo usa su propio preprocesador ajustado sobre su fold de train
sub_log_preds = np.zeros(len(X_submission_raw))

print(f'Generando predicciones de submission con {N_FOLDS} folds...\n')

for fold_idx, (prep_fold, model_fold) in enumerate(zip(fold_preps, fold_models)):
    X_sub_transformed  = prep_fold.transform(X_submission_raw)
    sub_log_preds     += model_fold.predict(X_sub_transformed) / N_FOLDS
    print(f'  Fold {fold_idx+1}/{N_FOLDS} OK')

sub_price_preds = np.expm1(sub_log_preds)

print(f'\nPredicciones de submission: {len(sub_price_preds):,} filas')
print(f'  Min    : ${sub_price_preds.min():>12,.0f}')
print(f'  Media  : ${sub_price_preds.mean():>12,.0f}')
print(f'  Mediana: ${np.median(sub_price_preds):>11,.0f}')
print(f'  Max    : ${sub_price_preds.max():>12,.0f}')

In [ ]:
submission_final = pd.DataFrame({
    'zpid'            : df_submission[ID_COL].values,
    'predicted_price' : sub_price_preds.round(2),
})

sub_out_path = SUBMISSIONS / f'{SUBMISSION_NAME}.csv'
submission_final.to_csv(sub_out_path, index=False)

# Verificacion
check_sub = pd.read_csv(sub_out_path)
assert list(check_sub.columns) == ['zpid', 'predicted_price'], 'Columnas incorrectas'
assert check_sub['predicted_price'].notna().all(), 'Hay NaN en predicciones'
assert (check_sub['predicted_price'] > 0).all(), 'Hay precios negativos o cero'

print(f'=== Submission Final ===')
print(f'  Archivo        : {sub_out_path.name}')
print(f'  Filas          : {len(check_sub):,}')
print(f'  Precio mediano : ${check_sub["predicted_price"].median():,.0f}')
print(f'  Precio minimo  : ${check_sub["predicted_price"].min():,.0f}')
print(f'  Precio maximo  : ${check_sub["predicted_price"].max():,.0f}')
print()
print('Submission OK')
print()
print(check_sub.head(10).to_string(index=False))

---
## 13. Exportacion para Metamodelo — Competition Test

Guarda las predicciones sobre el conjunto de competencia en `data/meta/comp_svm.csv`
para usarlas como feature de entrada en el metamodelo (stacking).

Las predicciones se guardan en **escala logaritmica** (`log_price`).

In [ ]:
meta_comp = pd.DataFrame({
    'zpid'     : df_submission[ID_COL].values,
    'svm_pred' : sub_log_preds.round(6),
})

meta_comp_path = META_DIR / 'comp_svm.csv'
meta_comp.to_csv(meta_comp_path, index=False)

print(f'Exportado: {meta_comp_path}')
print(f'Filas: {len(meta_comp):,}')
print(meta_comp.head())